# Kafka Demo

## Connect to the Kafka Broker Before Running the Notebook

Retrieve the SSH tunnel details and shared Kafka credential from the Canvas entry for this lab.
Do not save either credential in the repository or notebook.
Open a terminal and establish a foreground SSH tunnel to the Kafka server.

```
ssh -o ExitOnForwardFailure=yes -o ServerAliveInterval=60 -L 9092:localhost:<remote_port> <user>@<remote_server> -NT
```

Keep that terminal open while you run the notebook.
Open a second terminal and verify both the local tunnel and broker connection.

```bash
lsof -i :9092  # Should show an ssh process
kcat -F ~/.config/mlip-kafka.conf -b localhost:9092 -L  # See README for private config setup
```

Press <kbd>Ctrl</kbd>+<kbd>C</kbd> in the tunnel terminal when you finish the lab.

---

## Setup

The Codespace and DevContainer create `.venv` and install the notebook dependencies automatically.
Select `.venv/bin/python` as the notebook kernel if it is not already selected.
Use the following commands only when working outside the provided environment.
```
python -m venv <environment_name>
source <environment_name>/bin/activate  # On Windows: <environment_name>\Scripts\activate
```

Then install the requirements:
```
pip install -r requirements.txt
```
Or manually:
```
pip install kafka-python
```

In [ ]:
import os
from getpass import getpass
from datetime import datetime
from json import dumps, loads
from time import sleep
from random import randint
from kafka import KafkaConsumer, KafkaProducer
from typing import Dict, Any

# [TODO]: Fill in a unique identifier so your topic doesn't collide with others'
# Replace ... with your andrew_id as a string (e.g., "asmith") or any unique identifier
andrew_id = ...  # Example: andrew_id = "asmith"
topic = f"lab-kafka-{andrew_id}"
kafka_credential = getpass("Shared Kafka credential from Canvas: ")
kafka_auth = {
    "security_protocol": "SASL_PLAINTEXT",
    "sasl_mechanism": "PLAIN",
    "sasl_plain_username": "students",
    "sasl_plain_" + "password": kafka_credential,
}
print(f"Topic: {topic}")

### Producer Mode -> Writes Data to Broker

In [ ]:
# Below schema is for messages. You may change the city data if you wish but it is optional.
def make_city_data(city: str, temperature_f: str) -> Dict[str, Any]:
    return {
        "city": city,
        "timestamp": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "temperature_f": temperature_f, # temperature in fahrenheit
    }

In [ ]:
# Create a producer to write data to kafka
# Ref: https://kafka-python.readthedocs.io/en/master/apidoc/KafkaProducer.html

# [TODO]: Fill in the address of your Kafka bootstrap server
# [TODO]: Kafka expects messages as bytes. Explore the documentation and decide how to serialize Python dict objects into bytes.
# Hint: You may want to convert your Python dict → JSON string → UTF-8 bytes.

producer = KafkaProducer(bootstrap_servers=[...],
                        value_serializer=...,
                        **kafka_auth)

In [ ]:
# [TODO]: Add a few more examples of city data below
cities = [make_city_data("Pittsburgh" , 64), make_city_data(... , ...), make_city_data(... , ...)]

print("Writing to Kafka Broker")
for i in range(10):
    data = cities[randint(0,len(cities)-1)] # random selection
    producer.send(topic=topic, value=data)
    sleep(1)

producer.flush()
print(f"Data written to topic: {topic}")

### Consumer Mode -> Reads Data from Broker

In [ ]:
# Create a consumer to read data from kafka
# Ref: https://kafka-python.readthedocs.io/en/master/apidoc/KafkaConsumer.html

# [TODO]: Fill in the missing parameters:
#   1. First parameter: topic name (should match the topic you used in producer)
#   2. bootstrap_servers: same address you used in producer (e.g., ['localhost:9092'])
#   3. auto_offset_reset: try 'earliest' to read from beginning, 'latest' for new messages only
# Note: Since producer uses value_serializer, message.value is bytes. We decode and parse JSON.

consumer = KafkaConsumer(
    topic,  # [TODO]: Use your topic variable here
    bootstrap_servers=[...],  # [TODO]: Same bootstrap server as producer
    auto_offset_reset=...,  # [TODO]: Try 'earliest', 'latest', or 'none'
    group_id=topic,
    # Commit that an offset has been read
    enable_auto_commit=True,
    # How often to tell Kafka, an offset has been read
    auto_commit_interval_ms=1000,
    **kafka_auth
)

print('Reading Kafka Broker')
for message in consumer:
    # Producer serialized to JSON bytes, so we decode and parse
    message_str = message.value.decode('utf-8')
    message_dict = loads(message_str)
    print(message_dict)
    os.system(f"echo {message_str} >> kafka_log.csv")

# Use kcat!
It's a CLI (Command Line Interface). Previously known as kafkacat


Ref: https://docs.confluent.io/platform/current/app-development/kafkacat-usage.html

In [ ]:
# [TODO]: Use kcat to consume the first 5 messages from your topic. Use the -f flag to include the offset in the output.
# Example command structure:
# kcat -F ~/.config/mlip-kafka.conf -b localhost:9092 -t <your_topic_name> -C -o earliest -c 5 -f "%o: %s\n"
# 
# Where:
# - -b: broker address (same port as your SSH tunnel)
# - -t: your topic name (e.g., "lab-kafka-asmith")
# - -C: consumer mode
# - -o earliest: start from earliest offset
# - -c 5: consume 5 messages
# - -f "%o: %s\n": format to show offset and message
#
# Paste the output below and explain what the offset refers to.